# Aggregate model

This is the simplest DES model of our system, which can be run for the whole trust or per county - just change the CSV file of parameters provided.

## Imports

In [1]:
import pandas as pd
import plotly.express as px
from rich import print  # noqa: A004

from ambdes import (
    ArrivalConfig,
    Model,
    ModelConfig,
    Results,
    Runner,
    SimConfig,
    TimesConfig,
    plot_warm_up,
    run_warm_up_audit,
)

## Data sources

All parameters for the model come from CSV files. These currently contain synthetic parameters. There are three files:

In [2]:
pd.read_csv("../data/param_arrivals.csv")

,Unnamed: 0,C1,C2,C3,C4
0,monday,25,310,180,40
1,tuesday,24,295,170,38
2,wednesday,24,295,170,38
3,thursday,24,295,170,38
4,friday,25,305,178,40
5,saturday,28,340,200,45
6,sunday,27,330,195,43


In [3]:
pd.read_csv("../data/param_times.csv")

,time,type,C1,C2,C3,C4
0,travel_to_scene,mean,8,10,12,12
1,travel_to_scene,sd,5,5,5,5
2,on_scene,mean,44,46,48,50
3,on_scene,sd,5,5,5,5
4,travel_to_hospital,mean,8,10,12,12
5,travel_to_hospital,sd,5,5,5,5
6,handover,mean,15,22,40,45
7,handover,sd,5,5,5,5
8,wrap_up,mean,5,5,5,5
9,wrap_up,sd,2,2,2,2


In [4]:
pd.read_csv("../data/param_model.csv")

,parameter,value
0,resource_hours_per_week,52000
1,warm_up_period,500
2,data_collection_period,10080
3,n_reps,5


## Walk through the model

### Arrivals

The `ArrivalConfig` class loads `arrivals.csv` and uses it to define:

* **Patient arrivals:** a non-homogeneous Poisson process (NHPP) that varies by day of week. With a Poisson distribution, inter-arrival times are exponentially distributed.
* **Response categories:** assigned probabilistically based on observed proportions of each category.

In [5]:
arrival_config = ArrivalConfig(arrival_csv="../data/param_arrivals.csv")

# Parameters used in NHPP
display(arrival_config.nspp_df)

display(arrival_config.category_proportions)

,t,mean_iat
0,0,2.594595
1,1440,2.732448
2,2880,2.732448
3,4320,2.732448
4,5760,2.627737
5,7200,2.349103
6,8640,2.420168


C1    0.045478
C2    0.557674
C3    0.324411
C4    0.072437
dtype: float64

Our assumptions for arrivals are that:

* (a) Inter-arrival times vary by response category and day of week.
    * We agreed this in our discussion.
* (b) Proportion of each response category *do not* vary by day of week.
    * To check this assumption, we need to look at the proportion of C1 v.s., C2 v.s., C3 v.s., C4 by day of week...

In [6]:
# Checking proportion of each response category by day of week
display(arrival_config.proportion_df)
display(arrival_config.variation_df)

,C1,C2,C3,C4
monday,0.045045,0.558559,0.324324,0.072072
tuesday,0.045541,0.559772,0.322581,0.072106
wednesday,0.045541,0.559772,0.322581,0.072106
thursday,0.045541,0.559772,0.322581,0.072106
friday,0.045620,0.556569,0.324818,0.072993
saturday,0.045677,0.554649,0.326264,0.073409
sunday,0.045378,0.554622,0.327731,0.072269


,mean,min,max,range,sd
C1,0.045478,0.045045,0.045677,0.000632,0.000212
C2,0.557674,0.554622,0.559772,0.005150,0.002369
C3,0.324411,0.322581,0.327731,0.005150,0.002028
C4,0.072437,0.072072,0.073409,0.001337,0.000539


### Times

The `TimesConfig` class loads `times.csv`. Currently, times are all modelled as lognormal distributions, so a mean and standard deviation (SD) is provided for each.

We have assumed that each of these times *vary by response category* - though we should check this in the real data - the model could be simplified to just one time across categories if it doesn't vary.

In [7]:
times_config = TimesConfig(times_csv="../data/param_times.csv")
display(times_config.times_df)

,time,type,C1,C2,C3,C4
0,travel_to_scene,mean,8,10,12,12
1,travel_to_scene,sd,5,5,5,5
2,on_scene,mean,44,46,48,50
3,on_scene,sd,5,5,5,5
4,travel_to_hospital,mean,8,10,12,12
5,travel_to_hospital,sd,5,5,5,5
6,handover,mean,15,22,40,45
7,handover,sd,5,5,5,5
8,wrap_up,mean,5,5,5,5
9,wrap_up,sd,2,2,2,2


## Other parameters

The other parameters are imported by `ModelConfig`.

In [8]:
model_config = ModelConfig(param_csv="../data/param_model.csv")

### Configuration

The `SimConfig` class accepts instances of the config classes above, and uses them to create a single set of parameters ready for the model.

The distributions are all stored in `dist_config`, which follows a JSON format that can be accepted by the `sim-tools` `DistributionRegistry` class.

In [9]:
config = SimConfig(
    arrival_config=arrival_config,
    times_config=times_config,
    model_config=model_config,
)
print(config.__dict__)

{
    'dist_config': {
        'call_arrival': {
            'class_name': 'NSPPThinning',
            'params': {
                'data':       t  mean_iat
0     0  2.594595
1  1440  2.732448
2  2880  2.732448
3  4320  2.732448
4  5760  2.627737
5  7200  2.349103
6  8640  2.420168
            }
        },
        'call_category': {
            'class_name': 'DiscreteEmpirical',
            'params': {
                'values': Index(['C1', 'C2', 'C3', 'C4'], dtype='object'),
                'freq': array([0.04547757, 0.5576737 , 0.32441131, 0.07243742])
            }
        },
        'time_to_scene': {
            'C1': {'class_name': 'Lognormal', 'params': {'mean': 8, 'stdev': 5}},
            'C2': {'class_name': 'Lognormal', 'params': {'mean': 10, 'stdev': 5}},
            'C3': {'class_name': 'Lognormal', 'params': {'mean': 12, 'stdev': 5}},
            'C4': {'class_name': 'Lognormal', 'params': {'mean': 12, 'stdev': 5}}
        },
        'on_scene_time': {
            'C1': {'class_name': 'Lognormal', 'params': {'mean': 44, 'stdev': 5}},
            'C2': {'class_name': 'Lognormal', 'params': {'mean': 46, 'stdev': 5}},
            'C3': {'class_name': 'Lognormal', 'params': {'mean': 48, 'stdev': 5}},
            'C4': {'class_name': 'Lognormal', 'params': {'mean': 50, 'stdev': 5}}
        },
        'time_to_hospital': {
            'C1': {'class_name': 'Lognormal', 'params': {'mean': 8, 'stdev': 5}},
            'C2': {'class_name': 'Lognormal', 'params': {'mean': 10, 'stdev': 5}},
            'C3': {'class_name': 'Lognormal', 'params': {'mean': 12, 'stdev': 5}},
            'C4': {'class_name': 'Lognormal', 'params': {'mean': 12, 'stdev': 5}}
        },
        'handover_time': {
            'C1': {'class_name': 'Lognormal', 'params': {'mean': 15, 'stdev': 5}},
            'C2': {'class_name': 'Lognormal', 'params': {'mean': 22, 'stdev': 5}},
            'C3': {'class_name': 'Lognormal', 'params': {'mean': 40, 'stdev': 5}},
            'C4': {'class_name': 'Lognormal', 'params': {'mean': 45, 'stdev': 5}}
        },
        'wrap_up_time': {
            'C1': {'class_name': 'Lognormal', 'params': {'mean': 5, 'stdev': 2}},
            'C2': {'class_name': 'Lognormal', 'params': {'mean': 5, 'stdev': 2}},
            'C3': {'class_name': 'Lognormal', 'params': {'mean': 5, 'stdev': 2}},
            'C4': {'class_name': 'Lognormal', 'params': {'mean': 5, 'stdev': 2}}
        }
    },
    'n_ambulances': 310,
    'warm_up_period': 500,
    'data_collection_period': 10080,
    'n_reps': 5
}

### Model

The `Model` can be set-up by setting a run number and providing the `config` instance, then run by calling `run()`.

In [10]:
model = Model(run_number=0, config=config)
model.run()

### Logger

We record a log using the `vidigi` `EventLogger` class. This means it can work with `vidigi` to produce animations or process flow charts if desired.

We also have the attributes each patient stored in the model - for example, here, we can look at the patient with ID one in `model.patients` and in the `log`.

In [11]:
log = model.logger.to_dataframe()

# View patient with ID 1
print(model.patients[0].__dict__)
display(log[log["entity_id"] == 1])

{'patient_id': 1, 'category': 'C2', 'call_timestamp': 502.8679117477698, 'response_time': 7.706371045298511}

,entity_id,event_type,event,time,run_number,resource_id
0,1,arrival_departure,arrival,5.282029,0,NaN
1,1,queue,ambulance_wait_begins,5.282029,0,NaN
2,1,resource_use,ambulance_assigned,5.282029,0,1.0
140,1,resource_use_end,ambulance_available,124.944135,0,1.0
141,1,arrival_departure,depart,124.944135,0,NaN
895,1,arrival_departure,arrival,502.867912,0,NaN
896,1,queue,ambulance_wait_begins,502.867912,0,NaN
897,1,resource_use,ambulance_assigned,502.867912,0,194.0
1064,1,resource_use_end,ambulance_available,594.191491,0,194.0
1065,1,arrival_departure,depart,594.191491,0,NaN


## Results

The `Results` class accepts the run model instance and can calculate various performance measures.

In [12]:
Results(model).summary_df()

,category,n_patients,mean_response_time,run,mean_utilisation
0,C1,181.0,8.108579,0,NaN
1,C2,2229.0,9.925498,0,NaN
2,C3,1261.0,12.118782,0,NaN
3,C4,305.0,11.610874,0,NaN
4,all,NaN,NaN,0,0.151156


In [13]:
Results(model).utilisation_df()

,time,busy,interval_duration,utilisation
0,500.000000,229,1.923226,0.738710
1,501.923226,228,0.944686,0.735484
2,502.867912,229,1.063493,0.738710
3,503.931405,230,0.923537,0.741935
4,504.854942,231,0.018342,0.745161
...,...,...,...,...
7940,10573.211447,46,1.449287,0.148387
7941,10574.660734,47,0.188111,0.151613
7942,10574.848844,46,3.946366,0.148387
7943,10578.795210,45,0.311737,0.145161


### Runner

A `Runner` class is provided to run the model once or for multiple replications.

It also uses the `Results` class to calculate some performance measures.

In [14]:
runner = Runner(config)
results = runner.run_reps()

## Results

This is an example of running the model for one replication, for one week, with no warm-up period.

### Mean response time and utilisation

We can view by run and overall.

In [15]:
display(results["run"])

,category,n_patients,mean_response_time,run,mean_utilisation
0,C1,177.0,8.129865,0,NaN
1,C2,2114.0,9.890586,0,NaN
2,C3,1210.0,12.109557,0,NaN
3,C4,295.0,11.692520,0,NaN
4,all,NaN,NaN,0,0.149406
5,C1,167.0,8.600089,1,NaN
6,C2,2110.0,10.046693,1,NaN
7,C3,1396.0,11.928402,1,NaN
8,C4,278.0,12.272694,1,NaN
9,all,NaN,NaN,1,0.153128


In [16]:
display(results["overall"])

,category,mean_n_patients,mean_response_time,mean_utilisation
0,C1,175.4,8.275796,NaN
1,C2,2163.6,10.015794,NaN
2,C3,1275.0,12.068410,NaN
3,C4,282.4,11.929789,NaN
4,all,NaN,NaN,0.150445


### Example plot: distribution of response times by response category

This is an example taking results from a single model instance, and plotting distribution of response times observed during that run.

In [17]:
df = pd.DataFrame(
    {
        "response_time": [p.response_time for p in model.patients],
        "category": [p.category for p in model.patients],
    }
)

fig = px.histogram(
    df,
    x="response_time",
    facet_col="category",
    category_orders={"category": ["C1", "C2", "C3", "C4"]},
    labels={
        "response_time": "Response time (minutes)",
        "category": "Category",
    },
    title="Distribution of response times by category",
)

fig.layout.yaxis.title.text = "Number of patients"

fig.show()

## Determine appropriate warm-up period length

We can use the time series inspection approach to decide how long our warm-up period should be.

This requires recording performance measures at regular intervals. I have functions and classes to enable this analysis in `choose_warm_up.py`.

In [18]:
# Run audit
audit = run_warm_up_audit(config=config, interval=30, n_reps=5)

# Preview the dataframe produced
audit.head(20)

,time,category,metric,value,run
0,0,C1,response_time,NaN,0
1,0,C2,response_time,NaN,0
2,0,C3,response_time,NaN,0
3,0,C4,response_time,NaN,0
4,0,all,utilisation,0.000000,0
5,30,C1,response_time,NaN,0
6,30,C2,response_time,8.909374,0
7,30,C3,response_time,10.255449,0
8,30,C4,response_time,8.948742,0
9,30,all,utilisation,0.025009,0


### Example: C1 mean response time

In [19]:
plot_warm_up(audit=audit, metric="response_time", category="C1")

### Example: Mean utilisation

In [20]:
plot_warm_up(audit=audit, metric="utilisation")